In [ ]:
import sys
sys.path.append('..')

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import pandas as pd
import numpy as np

from src.config import PATHS
from src.viz import set_style, save_fig, annotate_source
from src.survey import weighted_mean, weighted_proportion, kish_design_effect, survey_summary_comparison, weighted_crosstab

set_style()

In [ ]:
df_h = pd.read_parquet(PATHS['analytical'] / 'df_hogar_analytic.parquet')
df_p = pd.read_parquet(PATHS['analytical'] / 'df_persona_analytic.parquet')

print(f'Hogares analíticos : {df_h.shape[0]:,} registros')
print(f'Personas analíticas: {df_p.shape[0]:,} registros')

## 10.2 Diagnóstico del Factor de Expansión ($FEX_C$) y Efecto de Diseño de Kish

In [ ]:
diag_h = kish_design_effect(df_h['fex_c'])
diag_p = kish_design_effect(df_p['fex_c'])

print(f'Nivel Hogar:')
print(f' Muestra observada (n) : {diag_h["n_muestra"]:,} hogares')
print(f' Población estimada (∑w_i) : {int(diag_h["suma_pesos"]):,} hogares colombianos')
print(f' Peso promedio (w̄) : {diag_h["media_peso"]:.2f}')
print(f' Desviación estándar peso : {diag_h["desv_est_peso"]:.2f}')
print(f' Coeficiente de variación (CV) : {diag_h["cv_peso"]:.2f}')
print(f' Efecto de diseño Kish (DEFF) : {diag_h["deff_kish"]:.2f}')
print(f' Tamaño efectivo (n_eff) : {int(diag_h["n_efectivo"]):,} hogares')

print(f'\nNivel Persona:')
print(f' Muestra observada (n) : {diag_p["n_muestra"]:,} personas')
print(f' Población estimada (∑w_i) : {int(diag_p["suma_pesos"]):,} habitantes')
print(f' Efecto de diseño Kish (DEFF) : {diag_p["deff_kish"]:.2f}')
print(f' Tamaño efectivo (n_eff) : {int(diag_p["n_efectivo"]):,} personas')

## 10.3 Comparación Sistemática: Estimación Muestral vs. Poblacional Ponderada

In [ ]:
vars_hogar = {
 'acceso_internet_hogar': 'Hogares: Acceso a Internet',
 'tiene_computador': 'Hogares: Tiene Computador',
 'tiene_internet_fijo': 'Hogares: Conexión Fija',
 'tiene_internet_movil': 'Hogares: Conexión Móvil',
 'pago_internet_hogar': 'Hogares: Pago mensual por Internet',
 'alta_dificultad_eco': 'Hogares: Alta Dificultad Económica',
 'es_urbano': 'Hogares: Dominio Urbano (Cabecera)'
}

tabla_comp_h = survey_summary_comparison(df_h, vars_hogar)
tabla_comp_h

In [ ]:
vars_persona = {
 'usa_internet': 'Personas: Usa Internet (últimos 12 meses)',
 'uso_diario_internet': 'Personas: Uso Diario intensivo',
 'tiene_celular': 'Personas: Posee Celular Propio',
 'tiene_educacion_superior': 'Personas: Educación Superior / Univ.',
 'es_urbano': 'Personas: Dominio Urbano'
}

tabla_comp_p = survey_summary_comparison(df_p, vars_persona)
tabla_comp_p

### Gráfico 12: Efecto del Factor de Expansión (FEX_C) — Comparación Muestral vs. Poblacional

In [ ]:
items = [
 ('Hogares: Conexión Móvil', df_h['tiene_internet_movil'].mean()*100, (df_h['tiene_internet_movil']*df_h['fex_c']).sum()/df_h['fex_c'].sum()*100),
 ('Hogares: Acceso a Internet', df_h['acceso_internet_hogar'].mean()*100, (df_h['acceso_internet_hogar']*df_h['fex_c']).sum()/df_h['fex_c'].sum()*100),
 ('Hogares: Conexión Fija', df_h['tiene_internet_fijo'].mean()*100, (df_h['tiene_internet_fijo']*df_h['fex_c']).sum()/df_h['fex_c'].sum()*100),
 ('Hogares: Tiene Computador', df_h['tiene_computador'].mean()*100, (df_h['tiene_computador']*df_h['fex_c']).sum()/df_h['fex_c'].sum()*100),
 ('Hogares: Dominio Urbano', df_h['es_urbano'].mean()*100, (df_h['es_urbano']*df_h['fex_c']).sum()/df_h['fex_c'].sum()*100),
 ('Personas: Usa Internet', df_p['usa_internet'].mean()*100, (df_p['usa_internet']*df_p['fex_c']).sum()/df_p['fex_c'].sum()*100),
 ('Personas: Uso Diario', df_p['uso_diario_internet'].mean()*100, (df_p['uso_diario_internet']*df_p['fex_c']).sum()/df_p['fex_c'].sum()*100),
 ('Personas: Celular Propio', df_p['tiene_celular'].mean()*100, (df_p['tiene_celular']*df_p['fex_c']).sum()/df_p['fex_c'].sum()*100),
]

df_plot = pd.DataFrame(items, columns=['Indicador', 'Sin Ponderar', 'Ponderado FEX_C']).iloc[::-1]

fig, ax = plt.subplots(figsize=(9, 5.5))
y_pos = np.arange(len(df_plot))
w = 0.35

b1 = ax.barh(y_pos + w/2, df_plot['Sin Ponderar'], height=w, label='Muestral Sin Ponderar (P̂)', color='#aec7e8', edgecolor='white')
b2 = ax.barh(y_pos - w/2, df_plot['Ponderado FEX_C'], height=w, label='Poblacional Ponderado FEX_C (P̂_w)', color='#1f77b4', edgecolor='white')

ax.set_yticks(y_pos)
ax.set_yticklabels(df_plot['Indicador'], fontsize=9.5)
ax.xaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_xlim(0, 110)
ax.set_xlabel('Tasa Estimada (%)', labelpad=8)
ax.set_title('Efecto del Factor de Expansión (FEX_C): Comparación Muestral vs. Poblacional', pad=15)
ax.legend(frameon=True, facecolor='white', loc='lower right')

for b in b1:
 val = b.get_width()
 ax.text(val + 0.8, b.get_y() + b.get_height()/2, f'{val:.1f}%', va='center', fontsize=8, color='#3182bd')
for b in b2:
 val = b.get_width()
 ax.text(val + 0.8, b.get_y() + b.get_height()/2, f'{val:.1f}%', va='center', fontsize=8, fontweight='bold', color='#1f77b4')

plt.tight_layout()
annotate_source(ax, 'Fuente: DANE — ECV 2024. Comparación de Horvitz-Thompson. Población: 18.5M hogares.')
save_fig(fig, 'fig12_comparacion_ponderada_no_ponderada')
plt.show()

## 10.4 Tablas de Contingencia Ponderadas con $FEX_C$

In [ ]:
tab_eco_pc_w = weighted_crosstab(df_h, 'sit_eco_label', 'tiene_computador', weight='fex_c', normalize='index')
tab_eco_pc_w.columns = ['Sin Computador (%)', 'Con Computador (%)']
print(tab_eco_pc_w.round(2))

df_p_valid = df_p[df_p['nivel_educ_agrupado'] != 'Menor de 5 años / No aplica']
tab_pedu_w = weighted_crosstab(df_p_valid, 'nivel_educ_agrupado', 'usa_internet', weight='fex_c', normalize='index')
tab_pedu_w.columns = ['No Usa Internet (%)', 'Usa Internet (%)']
print(tab_pedu_w.round(2))